# 🌿 Crop Disease Diagnostician — Full Pipeline
**Data download → Training → Convert → Evaluate → Download**

---
## ✅ Shuru karne se pehle:
1. **Runtime → Change runtime type → T4 GPU**
2. `kaggle.json` taiyar rakho
3. **Step 0A run karo PEHLE** — restart hogi, phir Step 0B se aage jao

---
## 💾 Google Drive Backup (Recommended)
Training ke baad model Google Drive mein save hoga automatically.
Agar Colab disconnect hua — `.h5` file Drive se download karke kaam chalega.

| Step | Time |
|---|---|
| Install + Data download | ~15 min |
| Model training | ~45 min |
| Convert + Download | ~15 min |
| **Total** | **~75 min** |

---
# 🛠️ STEP 0A — Fix Environment (SABSE PEHLE YAHI RUN KARO)
**Restart ke baad Step 0B se continue karo — yeh dubara mat chalana.**

In [ ]:
import importlib.metadata, subprocess, sys, os, time
try:
    version = importlib.metadata.version('ml_dtypes')
    major, minor = int(version.split('.')[0]), int(version.split('.')[1])
    print(f'ml_dtypes: {version}')
except Exception:
    major, minor = 0, 0

if (major, minor) < (0, 5):
    print(f'⚠️ Upgrading ml_dtypes...')
    subprocess.run([sys.executable, '-m', 'pip', 'install',
                    '--upgrade', '--quiet', 'ml_dtypes'], check=True)
    print('✅ Done! Restarting in 3 seconds...')
    print('   → Restart ke baad Step 0B se shuru karo!')
    time.sleep(3)
    os.kill(os.getpid(), 9)
else:
    print(f'✅ ml_dtypes {version} OK — Step 0B pe jao!')

---
# 📦 STEP 0B — Install Libraries
▶ **Restart ke baad yahaan se shuru karo**

In [ ]:
import importlib.metadata
version = importlib.metadata.version('ml_dtypes')
major, minor = int(version.split('.')[0]), int(version.split('.')[1])
assert (major, minor) >= (0, 5), \
    f'ml_dtypes {version} still old! Step 0A phir run karo aur restart karo.'
print(f'✅ ml_dtypes {version} OK')
!pip install -q kaggle seaborn scikit-learn
print('✅ Libraries installed!')

---
# 💾 STEP 0C — Mount Google Drive (Recommended)
Model Drive mein save hoga — agar Colab disconnect hua toh bhi safe rahega.

Skip karna ho toh seedha Step 1 pe jao.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Folder where model will be saved on Drive
DRIVE_SAVE_DIR = '/content/drive/MyDrive/CropDiseaseDiagnostician'
import os
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

print(f'✅ Drive mounted!')
print(f'   Model Drive pe save hoga: {DRIVE_SAVE_DIR}')
print(f'   Training khatam hone par automatically save hoga.')

USE_DRIVE = True

---
# 🔑 STEP 1 — Imports + Upload kaggle.json

In [ ]:
import os, json, shutil, random
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from google.colab import files

# Drive flag (False if Step 0C was skipped)
try: USE_DRIVE
except NameError: USE_DRIVE = False; DRIVE_SAVE_DIR = None

print(f'✅ TensorFlow: {tf.__version__}')
print(f'✅ GPU: {tf.config.list_physical_devices("GPU")}')
tf.keras.mixed_precision.set_global_policy('mixed_float16')
print(f'✅ Drive backup: {"ON → " + DRIVE_SAVE_DIR if USE_DRIVE else "OFF (skipped Step 0C)"}')

In [ ]:
print('kaggle.json upload karo:')
uploaded = files.upload()
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.copy('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('✅ Kaggle ready!')

---
# 🌱 STEP 2 — Download PlantVillage (~5 min)

In [ ]:
os.makedirs('/content/kaggle_download', exist_ok=True)
print('Downloading PlantVillage...')
!kaggle datasets download -d abdallahalidev/plantvillage-dataset \
    -p /content/kaggle_download --unzip

COLOR_DIR = None
for root, dirs, _ in os.walk('/content/kaggle_download'):
    if any('Tomato' in d for d in dirs) and any('Corn' in d for d in dirs):
        COLOR_DIR = root; break
if COLOR_DIR is None:
    for c in ['/content/kaggle_download/plantvillage dataset/color',
              '/content/kaggle_download/color']:
        if os.path.exists(c): COLOR_DIR = c; break
assert COLOR_DIR, '❌ Dataset not found!'
print(f'✅ Dataset: {COLOR_DIR}  ({len(os.listdir(COLOR_DIR))} classes)')

---
# 🗂️ STEP 3 — Organize 11 Classes

In [ ]:
IMG_SIZE=224; BATCH_SIZE=32; SEED=42
DATA_DIR='/content/data'; MODEL_DIR='/content/model'
PHASE1_EPOCHS=5; PHASE1_LR=1e-3
PHASE2_EPOCHS=15; PHASE2_LR=1e-4
UNFREEZE_FROM=100
os.makedirs(MODEL_DIR, exist_ok=True); random.seed(SEED)

TARGET_CLASSES = {
    'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot': 'corn_gray_leaf_spot',
    'Corn_(maize)___Common_rust_':                        'corn_common_rust',
    'Corn_(maize)___healthy':                             'corn_healthy',
    'Potato___Early_blight':                              'potato_early_blight',
    'Potato___Late_blight':                               'potato_late_blight',
    'Potato___healthy':                                   'potato_healthy',
    'Tomato___Bacterial_spot':                            'tomato_bacterial_spot',
    'Tomato___Early_blight':                              'tomato_early_blight',
    'Tomato___Late_blight':                               'tomato_late_blight',
    'Tomato___Leaf_Mold':                                 'tomato_leaf_mold',
    'Tomato___healthy':                                   'tomato_healthy',
}
LABELS_MAP=sorted(TARGET_CLASSES.values()); NUM_CLASSES=len(LABELS_MAP)
print(f'{NUM_CLASSES} classes ready!')
for i,k in enumerate(LABELS_MAP): print(f'  [{i}] {k}')

In [ ]:
print('Organizing train/val/test...')
split_stats={}; total_train=total_val=total_test=0
for folder,class_key in TARGET_CLASSES.items():
    src=os.path.join(COLOR_DIR,folder)
    imgs=[f for f in os.listdir(src) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    random.shuffle(imgs)
    n=len(imgs); nv=max(1,int(n*.1)); nt=max(1,int(n*.1)); nr=n-nv-nt
    for sp,si in [('train',imgs[:nr]),('val',imgs[nr:nr+nv]),('test',imgs[nr+nv:])]:
        d=os.path.join(DATA_DIR,sp,class_key); os.makedirs(d,exist_ok=True)
        for img in si: shutil.copy(os.path.join(src,img),os.path.join(d,img))
    split_stats[class_key]={'train':nr,'val':nv,'test':nt}
    total_train+=nr; total_val+=nv; total_test+=nt
    print(f'  {class_key:<30} train={nr} val={nv} test={nt}')

with open(f'{MODEL_DIR}/labels.json','w') as f: json.dump(LABELS_MAP,f,indent=2)
class_counts=[split_stats[k]['train'] for k in LABELS_MAP]
total_tr=sum(class_counts)
class_weights={i:total_tr/(NUM_CLASSES*c) for i,c in enumerate(class_counts)}
with open(f'{MODEL_DIR}/split_info.json','w') as f:
    json.dump({'labels':LABELS_MAP,'class_weights':{str(k):float(v) for k,v in class_weights.items()},
               'class_counts':class_counts},f,indent=2)
print(f'\n✅ Done! train:{total_train} val:{total_val} test:{total_test}')

---
# 🧠 STEP 4 — Pipeline + Model

In [ ]:
AUTOTUNE=tf.data.AUTOTUNE
assert sorted(os.listdir(f'{DATA_DIR}/train'))==LABELS_MAP

def make_dataset(split, augment=False):
    ds=tf.keras.utils.image_dataset_from_directory(
        f'{DATA_DIR}/{split}',image_size=(IMG_SIZE,IMG_SIZE),
        batch_size=None,label_mode='categorical',
        shuffle=(split=='train'),seed=SEED,class_names=LABELS_MAP)
    def preprocess(img,label):
        if augment:
            img=tf.image.random_flip_left_right(img)
            img=tf.image.random_flip_up_down(img)
            img=tf.image.random_brightness(img,0.2)
            img=tf.image.random_contrast(img,0.8,1.2)
            img=tf.image.random_saturation(img,0.8,1.2)
            img=tf.image.random_crop(img,[int(IMG_SIZE*.85),int(IMG_SIZE*.85),3])
            img=tf.image.resize(img,[IMG_SIZE,IMG_SIZE])
        return tf.keras.applications.mobilenet_v2.preprocess_input(img),label
    return ds.map(preprocess,num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

ds_train=make_dataset('train',augment=True)
ds_val  =make_dataset('val'  ,augment=False)
ds_test =make_dataset('test' ,augment=False)
print('✅ Pipeline ready!')

def build_model(num_classes,trainable_base=False):
    base=tf.keras.applications.MobileNetV2(
        input_shape=(IMG_SIZE,IMG_SIZE,3),include_top=False,weights='imagenet')
    base.trainable=trainable_base
    inp=tf.keras.Input(shape=(IMG_SIZE,IMG_SIZE,3))
    x=base(inp,training=trainable_base)
    x=tf.keras.layers.GlobalAveragePooling2D()(x)
    x=tf.keras.layers.BatchNormalization()(x)
    x=tf.keras.layers.Dense(256,activation='relu')(x)
    x=tf.keras.layers.Dropout(0.35)(x)
    x=tf.cast(x,tf.float32)
    out=tf.keras.layers.Dense(num_classes,activation='softmax',dtype='float32')(x)
    return tf.keras.Model(inp,out),base

model,base_model=build_model(NUM_CLASSES)
print(f'✅ Model built! Params: {sum(tf.size(w).numpy() for w in model.weights):,}')

---
# 🏋️ STEP 5 — Training Phase 1: Head Only (~10 min)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(PHASE1_LR),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy'])
hist1=model.fit(
    ds_train,epochs=PHASE1_EPOCHS,validation_data=ds_val,
    class_weight=class_weights,verbose=1,
    callbacks=[
        tf.keras.callbacks.ModelCheckpoint(f'{MODEL_DIR}/best_p1.keras',
            monitor='val_accuracy',save_best_only=True,verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',factor=0.5,patience=2,verbose=1)])
print(f'✅ Phase 1! Best val acc: {max(hist1.history["val_accuracy"])*100:.2f}%')

---
# 🔬 STEP 6 — Training Phase 2: Fine-Tune (~35 min)

In [ ]:
base_model.trainable=True
for l in base_model.layers[:UNFREEZE_FROM]: l.trainable=False
model.compile(
    optimizer=tf.keras.optimizers.Adam(PHASE2_LR),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy'])
hist2=model.fit(
    ds_train,initial_epoch=PHASE1_EPOCHS,
    epochs=PHASE1_EPOCHS+PHASE2_EPOCHS,
    validation_data=ds_val,class_weight=class_weights,verbose=1,
    callbacks=[
        tf.keras.callbacks.ModelCheckpoint(f'{MODEL_DIR}/crop_disease_model.keras',
            monitor='val_accuracy',save_best_only=True,verbose=1),
        tf.keras.callbacks.EarlyStopping(
            monitor='val_accuracy',patience=4,restore_best_weights=True,verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',factor=0.5,patience=2,verbose=1)])
model.save(f'{MODEL_DIR}/crop_disease_model.h5')
print(f'✅ Training done! Best val acc: {max(hist2.history["val_accuracy"])*100:.2f}%')

---
# 💾 STEP 6B — Save Model to Google Drive (BACKUP)
**Yeh cell training ke turant baad run hoga — model Drive pe safe ho jaayega.**

Agar Colab ab bhi disconnect kare toh `crop_disease_model.h5` Drive se download karo.

In [ ]:
if USE_DRIVE:
    print('💾 Saving model to Google Drive...')

    # Save model
    shutil.copy(f'{MODEL_DIR}/crop_disease_model.h5',
                f'{DRIVE_SAVE_DIR}/crop_disease_model.h5')

    # Save labels
    shutil.copy(f'{MODEL_DIR}/labels.json',
                f'{DRIVE_SAVE_DIR}/labels.json')

    print(f'\n✅ Model saved to Drive!')
    print(f'   Location: {DRIVE_SAVE_DIR}/')
    print(f'   Files: crop_disease_model.h5 + labels.json')
    print()
    print('   Agar Colab ab disconnect kare toh:')
    print('   → Drive kholo → CropDiseaseDiagnostician folder')
    print('   → crop_disease_model.h5 download karo')
    print('   → TFJS conversion locally ya new session mein kar sakte ho')
else:
    print('⚠️  Drive backup OFF (Step 0C skip kiya tha)')
    print('   Model sirf /content/model/ mein hai — disconnect se delete ho jaayega!')
    print('   Abhi manually download karo:')
    print('   Left sidebar → Files → content → model → crop_disease_model.h5 → Right click → Download')

---
# 📈 STEP 7 — Training Curves + Evaluate

In [ ]:
all_acc =hist1.history['accuracy']+hist2.history['accuracy']
all_vacc=hist1.history['val_accuracy']+hist2.history['val_accuracy']
all_loss=hist1.history['loss']+hist2.history['loss']
all_vloss=hist1.history['val_loss']+hist2.history['val_loss']
fig,(ax1,ax2)=plt.subplots(1,2,figsize=(14,5))
for ax,tr,vl,title in [(ax1,all_acc,all_vacc,'Accuracy'),(ax2,all_loss,all_vloss,'Loss')]:
    ax.plot(tr,label='Train',color='#16a34a',lw=2)
    ax.plot(vl,label='Val',color='#f59e0b',lw=2)
    ax.axvline(x=PHASE1_EPOCHS-.5,color='gray',ls='--',label='Fine-tune')
    ax.set_title(title); ax.legend(); ax.grid(alpha=0.3)
plt.suptitle('Training Curves',fontweight='bold'); plt.tight_layout()
plt.savefig(f'{MODEL_DIR}/training_curves.png',dpi=120); plt.show()

In [ ]:
y_true,y_pred=[],[]
for images,labels in ds_test:
    preds=model.predict(images,verbose=0)
    y_pred.extend(np.argmax(preds,axis=1).tolist())
    y_true.extend((np.argmax(labels.numpy(),axis=1) if len(labels.shape)>1 else labels.numpy()).tolist())
y_true=np.array(y_true); y_pred=np.array(y_pred)
top1_acc=accuracy_score(y_true,y_pred); macro_f1=f1_score(y_true,y_pred,average='macro')

print(f'\n{"="*50}')
print(f'  ⭐  RESULTS — Resume mein daalo!')
print(f'{"="*50}')
print(f'  Accuracy : {top1_acc*100:.2f}%')
print(f'  F1 Score : {macro_f1:.4f}')
print(f'  Samples  : {len(y_true)}')
print(f'{"="*50}')

report=classification_report(y_true,y_pred,target_names=LABELS_MAP,digits=4)
print(report)
with open(f'{MODEL_DIR}/classification_report.txt','w') as f:
    f.write(f'Accuracy:{top1_acc*100:.2f}%  F1:{macro_f1:.4f}\n\n{report}')

cm=confusion_matrix(y_true,y_pred); cm_norm=cm.astype('float')/cm.sum(axis=1)[:,np.newaxis]
short=['C-GLS','C-Rst','C-H','P-EB','P-LB','P-H','T-BS','T-EB','T-LB','T-LM','T-H']
fig,ax=plt.subplots(figsize=(11,9))
sns.heatmap(cm_norm,annot=True,fmt='.2f',cmap='Greens',
            xticklabels=short,yticklabels=short,ax=ax)
ax.set_title(f'Confusion Matrix  Acc:{top1_acc*100:.2f}%',fontweight='bold')
plt.tight_layout(); plt.savefig(f'{MODEL_DIR}/confusion_matrix.png',dpi=150); plt.show()

per_f1=f1_score(y_true,y_pred,average=None)
with open(f'{MODEL_DIR}/metrics.json','w') as f:
    json.dump({'test_accuracy':round(float(top1_acc),4),'macro_f1':round(float(macro_f1),4),
               'test_samples':int(len(y_true)),
               'per_class_f1':{k:round(float(v),4) for k,v in zip(LABELS_MAP,per_f1)}},f,indent=2)
print('✅ Evaluation done!')

---
# ⚡ STEP 8 — Convert to TFLite + TensorFlow.js

In [ ]:
calib_ds=tf.keras.utils.image_dataset_from_directory(
    f'{DATA_DIR}/val',image_size=(IMG_SIZE,IMG_SIZE),
    batch_size=None,shuffle=True,seed=42,class_names=LABELS_MAP)
calib_imgs=list(calib_ds.take(200).map(
    lambda img,_: tf.expand_dims(
        tf.keras.applications.mobilenet_v2.preprocess_input(tf.cast(img,tf.float32)),0)))
def representative_dataset():
    for s in calib_imgs: yield [s]

TFLITE_PATH=f'{MODEL_DIR}/crop_disease_model.tflite'
conv=tf.lite.TFLiteConverter.from_keras_model(model)
conv.optimizations=[tf.lite.Optimize.DEFAULT]
conv.representative_dataset=representative_dataset
conv.target_spec.supported_ops=[tf.lite.OpsSet.TFLITE_BUILTINS_INT8,tf.lite.OpsSet.TFLITE_BUILTINS]
conv.inference_input_type=tf.int8; conv.inference_output_type=tf.int8
print('Converting TFLite...')
tflite=conv.convert()
with open(TFLITE_PATH,'wb') as f: f.write(tflite)
print(f'✅ TFLite done! Size: {os.path.getsize(TFLITE_PATH)/1e6:.1f}MB')

In [ ]:
print('Installing tensorflowjs (training ke baad — no JAX conflict)...')
!pip install -q tensorflowjs
os.makedirs('/content/tfjs_model',exist_ok=True)
print('Converting to TensorFlow.js...')
!tensorflowjs_converter \
    --input_format=keras \
    --output_format=tfjs_layers_model \
    --quantize_float16 \
    /content/model/crop_disease_model.h5 \
    /content/tfjs_model/
shutil.copy(f'{MODEL_DIR}/labels.json','/content/tfjs_model/labels.json')
tfjs_mb=sum(os.path.getsize(os.path.join('/content/tfjs_model',f))
            for f in os.listdir('/content/tfjs_model'))/1e6
print(f'✅ TFJS done! Size: {tfjs_mb:.1f}MB')
!ls -lh /content/tfjs_model/

---
# 📥 STEP 9 — Download Everything

In [ ]:
# Also save TFJS model to Drive
if USE_DRIVE:
    tfjs_drive = f'{DRIVE_SAVE_DIR}/tfjs_model'
    if os.path.exists(tfjs_drive): shutil.rmtree(tfjs_drive)
    shutil.copytree('/content/tfjs_model', tfjs_drive)
    print(f'✅ TFJS model also saved to Drive: {tfjs_drive}')

shutil.make_archive('/content/tfjs_model_for_app','zip','/content/tfjs_model')
shutil.make_archive('/content/model_artifacts','zip',MODEL_DIR)

print('\n📦 Downloading 2 files:')
print('  1. tfjs_model_for_app.zip → unzip into app/public/model/')
print('  2. model_artifacts.zip    → TFLite + charts')

files.download('/content/tfjs_model_for_app.zip')
files.download('/content/model_artifacts.zip')

print(f'\n🎉 SAB KHATAM!  Accuracy:{top1_acc*100:.2f}%  F1:{macro_f1:.4f}')
print('Next: unzip tfjs_model_for_app.zip → copy to app/public/model/ → npm run dev')